# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Dataset ID: {md.id}")
print(f"Published: {md.datePublished}")
print(f"Version: {md.version}\n")
print(f"Available RecordSets: {md.recordSets}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their @id
record_sets = dataset.metadata.recordSets  # List of mlcroissant.RecordSet
print("RecordSets in Dataset:")
for rs in record_sets:
    print(f"- {rs.id}: {rs.name}")

# Print the fields for each record set
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs.name}' (ID: {rs.id}):")
    for field in rs.fields:
        print(f"  - {field.id} (type: {field.dataType})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use @id for every reference

dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    # records is a list of dicts or list of objects
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print DataFrame columns from the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in dataframes['{first_rs_id}']:")
    print(dataframes[first_rs_id].columns.tolist())
    print(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Filter and normalize 'Age' field, group by 'Sex' (if available)

# Find a numeric field, e.g., 'Age' (referenced by @id)

# Identify field @ids from the overview above
first_rs = dataset.metadata.recordSets[0]
numeric_fields = [f for f in first_rs.fields if f.dataType in ['schema:Float', 'schema:Integer', 'schema:Number']]
if numeric_fields:
    numeric_field_id = numeric_fields[0].id  # e.g., 'cr:field_Age'
else:
    numeric_field_id = None
    print("No numeric field found.")

# For demonstration, set threshold=50 for Age
if numeric_field_id:
    threshold = 50
    rs_id = first_rs.id
    df = dataframes[rs_id]
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a categorical/group field; e.g., 'Sex'
        group_field = None
        for fld in first_rs.fields:
            if 'sex' in fld.name.lower() or fld.dataType == 'schema:Text':
                group_field = fld.id
                break

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print(f"Field '{numeric_field_id}' not found in DataFrame columns.")
else:
    print("No numeric field available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If numeric_field_id and group_field are defined from above
if 'filtered_df' in locals() and filtered_df.shape[0] > 0 and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in filtered records (> {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field available, show comparison
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Used `mlcroissant` to load structured metadata and tabular records from the FAIR^2 colorectal cancer dataset.
- Identified available RecordSets and field `@id`s for consistent querying.
- Demonstrated filtering and normalization of a numeric field (e.g., Age), grouping by available fields (e.g., Sex), and visualizing distributions.
- This approach enables reproducible, schema-driven biomedical data exploration and can be extended for further statistical or ML analysis.